In [ ]:
import os
import re
import matplotlib.pyplot as plt

results_dir = "/home/m-sef/Documents/projects/cloudlab-auto-keda-cluster/experiments/memcached/results"

data = []
for entry in os.listdir(results_dir):
    parts = entry.split("_")
    if len(parts) != 2:
        continue
    update_frac, target_qps_str = parts
    target_qps = int(target_qps_str)
    if target_qps < 200000:  # only new high-QPS results
        continue
    leader_log = os.path.join(results_dir, entry, "leader.log")
    if not os.path.exists(leader_log):
        continue

    p99_read = p99_update = actual_qps = None
    with open(leader_log) as f:
        for line in f:
            if line.startswith("read"):
                p99_read = float(line.split()[8])
            elif line.startswith("update"):
                p99_update = float(line.split()[8])
            elif line.startswith("Total QPS"):
                m = re.search(r"Total QPS = ([\d.]+)", line)
                if m:
                    actual_qps = float(m.group(1))

    if p99_read is not None and p99_update is not None:
        data.append((target_qps, actual_qps, p99_read, p99_update))

data.sort(key=lambda x: x[0])
actual_qps_vals = [d[1] / 1000 for d in data]  # in thousands
p99_read_vals = [d[2] for d in data]
p99_update_vals = [d[3] for d in data]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(actual_qps_vals, p99_read_vals, marker="o", label="P99 Read latency")
ax.plot(actual_qps_vals, p99_update_vals, marker="s", label="P99 Update latency")

ax.set_xlabel("Actual QPS (thousands)")
ax.set_ylabel("P99 Latency (µs)")
ax.set_title("Memcached P99 Latency vs QPS (update fraction = 0.25)")
ax.legend()
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0f}K"))

plt.tight_layout()
plt.show()

for d in data:
    print(f"Target {d[0]//1000:4d}K (actual {d[1]/1000:6.1f}K QPS): read P99={d[2]:.1f}µs  update P99={d[3]:.1f}µs")
